In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType,DateType
from pyspark.sql.functions import trim, col

# Read from Bronze Table

In [0]:
df = spark.table("workspace.bronze.erp_loc_a101")

In [0]:
df.limit(10).display()

# Transformations

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

In [0]:
df = df.withColumn(
    "cid",
    F.when(col("cid").startswith("AW-"),
           F.expr("replace(cid, 'AW-', 'AW')"))
     .otherwise(col("cid"))
)

In [0]:
RENAME_MAP = {
    "cid": "customer_number",
    "cntry": "country",
    
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df = df.withColumn(
    "country",
    F.when(col("country") == "DE", "Germany")
     .when(col("country").isin("US", "USA"), "United States")
     .when((col("country") == "") | col("country").isNull(), "n/a")
     .otherwise(col("country"))
)

In [0]:
df.limit(10).display()

# put in the Silver table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_loc")

In [0]:
%sql
SELECT * FROM workspace.silver.erp_loc LIMIT 10